In [4]:
## Cell 1: Import Libraries and Read Bronze Data

from pyspark.sql.functions import col, year, month, sum as spark_sum, count, round

storage_account = "stsynbootcampvishnu"

bronze_base = f"abfss://processed@{storage_account}.dfs.core.windows.net/bronze"
silver_base = f"abfss://processed@{storage_account}.dfs.core.windows.net/silver"
gold_base = f"abfss://processed@{storage_account}.dfs.core.windows.net/gold"

customers_path = f"{bronze_base}/customers.csv"
accounts_path = f"{bronze_base}/accounts.csv"
transactions_path = f"{bronze_base}/transactions.csv"

customers_df = spark.read.option("header", True).csv(customers_path)
accounts_df = spark.read.option("header", True).csv(accounts_path)
transactions_df = spark.read.option("header", True).csv(transactions_path)

display(customers_df)
display(accounts_df)
display(transactions_df)

StatementMeta(sparkpool1, 1, 2, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 2231382a-3ee3-4f12-98b0-eb223ff25126)

SynapseWidget(Synapse.DataFrame, b5ad6f69-34bb-4bb1-bd83-ed86c7cfabb5)

SynapseWidget(Synapse.DataFrame, faed1cbd-03ef-4b45-b5a9-a4c7a7785a1f)

In [5]:
## Cell 2: Clean and Prepare Data
customers_clean = customers_df.dropDuplicates(["customer_id"]).dropna(subset=["customer_id"])

accounts_clean = accounts_df.dropDuplicates(["account_id"]).dropna(subset=["account_id", "customer_id"])

transactions_clean = (
    transactions_df
    .dropDuplicates(["transaction_id"])
    .dropna(subset=["transaction_id", "account_id", "customer_id", "transaction_date", "amount"])
    .withColumn("transaction_date", col("transaction_date").cast("date"))
    .withColumn("amount", col("amount").cast("double"))
    .withColumn("ingestion_timestamp", col("ingestion_timestamp").cast("timestamp"))
)

StatementMeta(sparkpool1, 1, 3, Finished, Available, Finished, False)

In [6]:
## Cell 3: Save Silver Layer
customers_clean.write.mode("overwrite").parquet(f"{silver_base}/customers")
accounts_clean.write.mode("overwrite").parquet(f"{silver_base}/accounts")
transactions_clean.write.mode("overwrite").partitionBy("transaction_date").parquet(f"{silver_base}/transactions")

StatementMeta(sparkpool1, 1, 4, Finished, Available, Finished, False)

In [8]:
## Cell 4: Create Gold Reporting Table
customer_transaction_gold = (
    transactions_clean.alias("t")
    .join(customers_clean.alias("c"), col("t.customer_id") == col("c.customer_id"), "left")
    .join(accounts_clean.alias("a"), col("t.account_id") == col("a.account_id"), "left")
    .select(
        col("t.transaction_id"),
        col("t.transaction_date"),
        year(col("t.transaction_date")).alias("transaction_year"),
        month(col("t.transaction_date")).alias("transaction_month"),
        col("t.transaction_type"),
        col("t.amount"),
        col("t.merchant_category"),
        col("t.payment_method"),
        col("t.transaction_status"),
        col("t.transaction_location"),
        col("c.customer_id"),
        col("c.customer_name"),
        col("c.city"),
        col("c.province"),
        col("c.customer_segment"),
        col("a.account_id"),
        col("a.account_type")
    )
)

display(customer_transaction_gold)

StatementMeta(sparkpool1, 1, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, aa7f9067-5ccb-481d-acee-f8b5287a66b1)

In [9]:
## Cell 5: Save Gold Layer
customer_transaction_gold.write.mode("overwrite").partitionBy("transaction_year", "transaction_month").parquet(f"{gold_base}/customer_transaction_gold")

StatementMeta(sparkpool1, 1, 7, Finished, Available, Finished, False)

In [10]:
## Cell 6: Validate Row Counts
raw_transaction_count = transactions_df.count()
silver_transaction_count = transactions_clean.count()
gold_transaction_count = customer_transaction_gold.count()

print("Raw transaction count:", raw_transaction_count)
print("Silver transaction count:", silver_transaction_count)
print("Gold transaction count:", gold_transaction_count)

StatementMeta(sparkpool1, 1, 8, Finished, Available, Finished, False)

Raw transaction count: 60
Silver transaction count: 60
Gold transaction count: 60


In [11]:
## Cell 7: Check Data Completeness
transactions_clean.select([
    count(col(c)).alias(c) for c in transactions_clean.columns
]).show()

StatementMeta(sparkpool1, 1, 9, Finished, Available, Finished, False)

+--------------+----------+-----------+----------------+----------------+------+-----------------+--------------+------------------+--------------------+-------------------+
|transaction_id|account_id|customer_id|transaction_date|transaction_type|amount|merchant_category|payment_method|transaction_status|transaction_location|ingestion_timestamp|
+--------------+----------+-----------+----------------+----------------+------+-----------------+--------------+------------------+--------------------+-------------------+
|            60|        60|         60|              60|              60|    60|               60|            60|                60|                  60|                 60|
+--------------+----------+-----------+----------------+----------------+------+-----------------+--------------+------------------+--------------------+-------------------+



In [12]:
## Cell 8: Customer Segment Summary
customer_transaction_gold.groupBy("customer_segment").agg(
    count("*").alias("total_transactions"),
    round(spark_sum("amount"), 2).alias("total_amount")
).show()

StatementMeta(sparkpool1, 1, 10, Finished, Available, Finished, False)

+----------------+------------------+------------+
|customer_segment|total_transactions|total_amount|
+----------------+------------------+------------+
|         Premium|                13|    31578.39|
|        Business|                19|    24679.25|
|          Retail|                28|    63209.89|
+----------------+------------------+------------+



In [13]:
## Cell 9: Read Gold Data
gold_path = f"{gold_base}/customer_transaction_gold"

gold_df = spark.read.parquet(gold_path)

display(gold_df)

StatementMeta(sparkpool1, 1, 11, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 4d263d75-7b9c-43d8-a612-c0cc7391e2ab)

In [14]:
## Cell 10: Customer Segment Report
from pyspark.sql.functions import count, sum as spark_sum, round

segment_report = (
    gold_df
    .groupBy("customer_segment")
    .agg(
        count("*").alias("total_transactions"),
        round(spark_sum("amount"), 2).alias("total_amount")
    )
    .orderBy("customer_segment")
)

display(segment_report)

StatementMeta(sparkpool1, 1, 12, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, b1108879-c076-4f7d-abf8-2f2f7021a922)

In [15]:
## Cell 11: Monthly Transaction Report
monthly_report = (
    gold_df
    .groupBy("transaction_year", "transaction_month")
    .agg(
        count("*").alias("total_transactions"),
        round(spark_sum("amount"), 2).alias("total_amount")
    )
    .orderBy("transaction_year", "transaction_month")
)

display(monthly_report)

StatementMeta(sparkpool1, 1, 13, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 1b6d70c1-9c6c-4c69-87d5-74ee87fd9ac9)

In [16]:
## Cell 12: Transaction Type Report
transaction_type_report = (
    gold_df
    .groupBy("transaction_type")
    .agg(
        count("*").alias("total_transactions"),
        round(spark_sum("amount"), 2).alias("total_amount")
    )
)

display(transaction_type_report)

StatementMeta(sparkpool1, 1, 14, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 9cae2118-3883-49a8-b2d9-86e30adc416c)

In [17]:
## Cell 13: Payment Method Report
payment_method_report = (
    gold_df
    .groupBy("payment_method")
    .agg(
        count("*").alias("total_transactions"),
        round(spark_sum("amount"), 2).alias("total_amount")
    )
    .orderBy("payment_method")
)

display(payment_method_report)

StatementMeta(sparkpool1, 1, 15, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 5b476bf6-8fc7-4372-b71c-27d167c88fe5)

In [18]:
## Cell 14: Final Count Check
raw_transaction_count = transactions_df.count()
silver_transaction_count = transactions_clean.count()
gold_transaction_count = gold_df.count()

print("Raw transaction count:", raw_transaction_count)
print("Silver transaction count:", silver_transaction_count)
print("Gold transaction count:", gold_transaction_count)


StatementMeta(sparkpool1, 1, 16, Finished, Available, Finished, False)

Raw transaction count: 60
Silver transaction count: 60
Gold transaction count: 60
Validation Passed: Raw, Silver, and Gold transaction counts match.


In [19]:
## Cell 15: Print Gold Schema
gold_df.printSchema()

StatementMeta(sparkpool1, 1, 17, Finished, Available, Finished, False)

root
 |-- transaction_id: string (nullable = true)
 |-- transaction_date: date (nullable = true)
 |-- transaction_type: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- merchant_category: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- transaction_status: string (nullable = true)
 |-- transaction_location: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- province: string (nullable = true)
 |-- customer_segment: string (nullable = true)
 |-- account_id: string (nullable = true)
 |-- account_type: string (nullable = true)
 |-- transaction_year: integer (nullable = true)
 |-- transaction_month: integer (nullable = true)



In [21]:
## Cell 17: Verify Gold Folder Structure
mssparkutils.fs.ls(f"{gold_base}/customer_transaction_gold")

StatementMeta(sparkpool1, 1, 19, Finished, Available, Finished, False)

[FileInfo(path=abfss://processed@stsynbootcampvishnu.dfs.core.windows.net/gold/customer_transaction_gold/_SUCCESS, name=_SUCCESS, size=0),
 FileInfo(path=abfss://processed@stsynbootcampvishnu.dfs.core.windows.net/gold/customer_transaction_gold/transaction_year=2025, name=transaction_year=2025, size=0)]